In [1]:
from anthropic import Anthropic
from dotenv import load_dotenv
import os
import json

In [2]:
load_dotenv(override=True)

client = Anthropic()
model = "claude-sonnet-5"

In [3]:
CLASSIFY_INTENT = {
    "name": "classify_intent",
    "description": "Record the classified intent of a customer support message.",
    "input_schema": {
        "type": "object",
        "properties": {
            "intent": {
                "type": "string",
                "enum": ["refund_request", "order_status",
                         "billing_issue", "product_question", "other"],
                "description": (
                    "The customer's primary intent. Use 'other' only when the "
                    "message fits none of the specific categories."
                ),
            },
        },
        "required": ["intent"],
    },
}

SYSTEM = (
    "You classify inbound customer support messages by intent. "
    "Classify the customer's message using the classify_intent tool."
)


def classify_intent(customer_message: str) -> str:
    message = client.messages.create(
        model=model,
        max_tokens=1024,
        system=SYSTEM,
        output_config={"effort": "low"},
        tools=[CLASSIFY_INTENT],
        tool_choice={"type": "any"},
        messages=[{
            "role": "user",
            "content": f"<customer_message>\n{customer_message}\n</customer_message>",
        }],
    )
    for block in message.content:
        if block.type == "tool_use":
            return block.input["intent"]
    raise RuntimeError(f"no tool_use block; stop_reason={message.stop_reason}")

In [4]:
test_cases = [
    {
        "input": "I want to return my shoes.  They arrived damaged.",
        "expected_intent": "refund_request"
    },
    {
        "input": "Where is my order?  It was supposed to arrive yesterday.",
        "expected_intent": "order_status"
    },
    {
        "input": "I was charged twice for the same order.",
        "expected_intent": "billing_issue"
    },
    {
        "input": "Does this jacket run true to size?",
        "expected_intent": "product_question"
    },
    {
        "input": "Are you open on Sundays?",
        "expected_intent": "other"
    }
]

In [5]:
def run_eval(cases):
    passed = 0
    for case in cases:
        got = classify_intent(case["input"])
        ok = got == case["expected_intent"]
        passed += ok
        status = "PASS" if ok else "FAIL"
        print(f"{status}  expected={case['expected_intent']:<16} got={got:<16} {case['input'][:40]}")
    print(f"\naccuracy: {passed}/{len(cases)}")

In [6]:
run_eval(test_cases)

PASS  expected=refund_request   got=refund_request   I want to return my shoes.  They arrived
PASS  expected=order_status     got=order_status     Where is my order?  It was supposed to a
PASS  expected=billing_issue    got=billing_issue    I was charged twice for the same order.
PASS  expected=product_question got=product_question Does this jacket run true to size?
PASS  expected=other            got=other            Are you open on Sundays?

accuracy: 5/5
